# 03. Criterion Validity Against Self-Reported Emotion (Appendix E.3; Table A3)


In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load self-reports and model scores

In [2]:
SR = pd.read_csv(os.path.join(C.REPRO, "selfreport_clean.csv"))[["pid", "self_sympathy", "self_inspired"]]
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv"))
LAB = {e: g.drop(columns="exp").merge(SR, on="pid") for e, g in ALL.groupby("exp") if e in C.MLLM_EXPS}
for d in LAB.values(): d["S_hi"] = d.true.map(C.SYM_HI); d["I_hi"] = d.true.map(C.INS_HI)
print({e: len(LAB[e]) for e in LAB})

{1: 95, 2: 99, 3: 99, 4: 99, 5: 99, 6: 99, 7: 99, 8: 99, 9: 98, 16: 99}


## 2. Zero-order convergent and discriminant correlations

In [3]:
rows = []
PAIRS = [("symp", "self_sympathy", "convergent"), ("insp", "self_inspired", "convergent"), ("symp", "self_inspired", "discriminant"), ("insp", "self_sympathy", "discriminant")]
for e, d in LAB.items():
    for ms, sr, kind in PAIRS:
        rho, p, lo, hi = C.spearman_with_ci(d[ms].values, d[sr].values)
        rows.append(dict(exp=e, configuration=C.NAMES[e], analysis="zero_order", kind=kind, model_score=ms, self_report=sr, N=len(d), rho=rho, ci_lo=lo, ci_hi=hi, p=p))
ZO = pd.DataFrame(rows)
ZO[ZO.kind == "convergent"].pivot(index="exp", columns="model_score", values=["rho", "p"]).round(3)

Out[3]: 
               rho             p       
model_score   insp   symp   insp   symp
exp                                    
1            0.118  0.106  0.254  0.307
2            0.196  0.501  0.052  0.000
3            0.543  0.574  0.000  0.000
4            0.623  0.680  0.000  0.000
5            0.576  0.678  0.000  0.000
6            0.647  0.647  0.000  0.000
7            0.583  0.516  0.000  0.000
8            0.536  0.598  0.000  0.000
9            0.541  0.751  0.000  0.000
16           0.449  0.398  0.000  0.000


## 3. Condition-controlled (partial) correlations and design dependence

In [4]:
for e, d in LAB.items():
    Z = d[["S_hi", "I_hi"]].values.astype(float)
    for ms, sr, kind in PAIRS[:2]:
        r, p, lo, hi, n = C.partial_spearman_boot(d[ms].values, d[sr].values, Z)
        rows.append(dict(exp=e, configuration=C.NAMES[e], analysis="partial_condition_controlled", kind=kind, model_score=ms, self_report=sr, N=n, rho=r, ci_lo=lo, ci_hi=hi, p=p))
    for ms, dummy in [("symp", "S_hi"), ("insp", "I_hi")]:
        rho, p, lo, hi = C.spearman_with_ci(d[ms].values, d[dummy].values)
        rows.append(dict(exp=e, configuration=C.NAMES[e], analysis="model_score_vs_design_dummy", kind="convergent", model_score=ms, self_report=dummy, N=len(d), rho=rho, ci_lo=lo, ci_hi=hi, p=p))
    for sr, dummy in [("self_sympathy", "S_hi"), ("self_inspired", "I_hi")]:
        rho, p, lo, hi = C.spearman_with_ci(d[sr].values, d[dummy].values)
        rows.append(dict(exp=e, configuration=C.NAMES[e], analysis="self_report_vs_design_dummy", kind="convergent", model_score=dummy, self_report=sr, N=len(d), rho=rho, ci_lo=lo, ci_hi=hi, p=p))
SRV = pd.DataFrame(rows); SRV.to_csv(os.path.join(C.OUT, "selfreport_validity.csv"), index=False)
SRV[SRV.analysis == "partial_condition_controlled"].pivot(index="exp", columns="model_score", values=["rho", "ci_lo", "ci_hi"]).round(2)

Out[4]: 
              rho       ci_lo       ci_hi      
model_score  insp  symp  insp  symp  insp  symp
exp                                            
1           -0.06 -0.15 -0.16 -0.34  0.01  0.05
2           -0.03  0.09 -0.24 -0.11  0.18  0.29
3            0.13  0.09 -0.08 -0.15  0.33  0.32
4            0.27  0.15  0.04 -0.12  0.44  0.38
5            0.19  0.22 -0.02 -0.02  0.39  0.44
6            0.30  0.10  0.07 -0.12  0.51  0.32
7            0.24  0.25  0.02  0.04  0.44  0.45
8            0.14  0.20 -0.08 -0.03  0.35  0.41
9            0.07  0.28 -0.16  0.03  0.29  0.49
16           0.17 -0.03 -0.06 -0.26  0.38  0.20


## 4. Verification against the frozen outputs

Point estimates and *p* values reproduce exactly; the bootstrap interval bounds differ by at most .011 because the resampling order of the original run is not recoverable.

In [5]:
fz = C.load_frozen("selfreport_validity_frozen.csv")
fz = fz[fz.analysis.isin(SRV.analysis.unique())]
print(C.compare(SRV, fz, ["exp", "analysis", "kind", "model_score", "self_report"], ["N", "rho", "ci_lo", "ci_hi", "p"], atol=0.02).to_string(index=False))

column  n_compared  max_abs_diff  within_tolerance
     N          60  0.000000e+00              True
   rho          60  2.775558e-16              True
 ci_lo          60  1.105944e-02              True
 ci_hi          60  9.031977e-03              True
     p          60  5.551115e-16              True
